> 📌 このノートブックは [Learn-Prompt-Hacking](https://github.com/TrustAI-laboratory/Learn-Prompt-Hacking) の日本語版です。

# セットアップ

LLM API を使うための初期設定を行います。  
事前に [00_Setup.ipynb](00_Setup.ipynb) を完了してください。

# 設定

In [ ]:
%pip install -q openai

In [ ]:
import sys; sys.path.insert(0, "..")
from utils.llm_client import call_llm_and_print as call_GPT, md_print

# AI の頭の中を理解する

すべての機械学習アルゴリズムと同様に、LLM にも調整可能なハイパーパラメータがあります。最も一般的に使用されるものには以下があります:

* **Temperature（温度）**: LLM 出力のランダム性に影響するハイパーパラメータ。高い温度は予測不可能で創造的な結果を生み、低い温度は一般的で保守的な出力を生成します。

* **Top p**: LLM 出力の最も可能性の高いトークンを選択するための確率閾値を設定するハイパーパラメータ。累積確率が閾値を超える上位トークンを考慮することで、より多様な出力を生成できます。例えば、top p を 0.9 に設定すると、確率質量の 90% を占める最も可能性の高い単語からのみランダムサンプリングを行います。高い値は候補が少なく、低い値は候補が多くなります。

* **Frequency penalty（頻度ペナルティ）**: 言語モデルの生成過程に適用されるペナルティ項で、単語やフレーズの過度な繰り返しを避けるためのものです。頻度ペナルティを追加することで、モデルはより多様で繰り返しの少ないテキストを生成するよう促されます。

* **Presence penalty（存在ペナルティ）**: 特定の単語やフレーズの生成を抑制する別のペナルティ項。特定の単語やフレーズに高いペナルティ値を割り当てることで、モデルがそれらを生成テキストで使用する可能性が低くなります。これは、機密性の高い分野や制限された分野など、特定の単語やフレーズを避ける必要があるテキスト生成に有用です。

これらのハイパーパラメータの値とモデル出力の関係を理解することで、タスクとプロンプトセットに適した組み合わせを活用して結果を最適化できます。

# プロンプティングとは？

人間は AI にタスクを実行するよう指示できます。プロンプトを使って AI にタスクを指示することを**プロンプティング**と呼びます。

ここでは、非常に人気のある大規模言語モデル（LLM）である ChatGPT を使ってプロンプティングを探求します。ChatGPT はテキストを理解し生成することができ、OpenAI によって開発されました。現在、最も扱いやすい生成 AI の一つです。

# LLM API を使ったプロンプティング

OpenAI のモデルだけが LLM ではありませんが、API によるアクセスの容易さとモデル品質の高さから、このコース全体でプロンプティング技法を紹介するのに適しています。

上のセットアップセルで読み込んだ `call_GPT` 関数を使って、プロンプトからテキストを生成します。

いくつかのプロンプトを試してみましょう。

In [ ]:
# --- 元の英語プロンプト（参考用） ---
# call_GPT(prompt="Write an essay explaining what AI is.")

# AI についてのエッセイを書かせる
call_GPT(prompt="AI とは何かを説明するエッセイを書いてください。")

自然言語を通じて AI にタスクを指示するプロセスを**プロンプティング**と呼びます。AI に何をすべきかを伝えれば、それを実行してくれます！

プロンプトは、シンプルな指示や質問から、大量のテキストを含む複雑なものまでさまざまです。

In [ ]:
# --- 元の英語プロンプト（参考用） ---
# summarize_prompt="""
# It is very rare for snow to fall in the U.S. state of Florida, especially in the
# central and southern portions of the state. With the exception of the far northern
# areas of the state, most of the major cities in Florida have never recorded measurable
# snowfall, though trace amounts have been recorded, or flurries in the air observed
# few times each century. According to the National Weather Service, in the Florida
# Keys and Key West there is no known occurrence of snow flurries since the European
# colonization of the region more than 300 years ago. In Miami, Fort Lauderdale, and
# Palm Beach there has been only one known report of snow flurries observed in the air
# in more than 200 years; this occurred in January 1977. In any event, Miami, Fort
# Lauderdale, and Palm Beach have not seen snow flurries before or since this 1977 event.
#
# Summarize this paragraph in a single sentence:
# """

# 要約プロンプト
summarize_prompt="""
アメリカのフロリダ州で雪が降ることは非常に稀であり、特に州の中部および南部ではほとんど見られません。州の最北部を除けば、フロリダの主要都市のほとんどで計測可能な降雪が記録されたことはなく、わずかな痕跡が記録されたり、数世紀に数回空中に風花が観測された程度です。アメリカ国立気象局によると、フロリダキーズおよびキーウェストでは、300年以上前のヨーロッパ人による植民地化以降、風花が発生した記録はありません。マイアミ、フォートローダーデール、パームビーチでは、200年以上の間に空中で風花が観測された報告はわずか1件のみで、これは1977年1月に発生しました。いずれにしても、マイアミ、フォートローダーデール、パームビーチでは、この1977年の事象の前後で風花は観測されていません。

この段落を1文で要約してください:
"""

call_GPT(summarize_prompt)

# Instruction Prompting（指示プロンプティング）


最初のプロンプティング技法として、**Instruction Prompting（指示プロンプティング）** を取り上げます。生成 AI にはもっと複雑なタスクも指示できます。

いくつかの複雑な指示に従わせてみましょう：

## 1、名前の解析（Name Parsing）


名前データを収集する際によくある問題は、人によって名前のフォーマットが異なることです。Mrs. や Jr. が付いていたり、姓と名の順序が逆だったりします。以前は、このようなデータのクリーニングは退屈な手作業でした。今では、シンプルなプロンプトで完全に自動化できます。

In [ ]:
# --- 元の英語プロンプト（参考用） ---
# instruction_prompt_example1 ="""
# A user has input their first and last name into a form. We don't know in which order
# their first/last name is, but we need it to be in the format 'Last, First'. Convert the following:
#
# john doe
# """

instruction_prompt_example1 ="""
ユーザーがフォームに姓と名を入力しました。姓と名がどの順番で入力されたか分かりませんが、
「姓, 名」の形式にする必要があります。以下を変換してください：

山田 太郎
"""

call_GPT(instruction_prompt_example1)

## 2、個人識別情報の除去（PII Removal）

PII（個人識別情報）の除去も重要なタスクです。機密文書を公開する前に、企業や政府は手作業で情報を墨消しすることがあります。生成 AI を使えば PII を自動的に除去でき、膨大な人的作業を省くことができます。

In [ ]:
# --- 元の英語プロンプト（参考用） ---
# instruction_prompt_example2 ="""
# Read the following sales email. Remove any personally identifiable information (PII),
# and replace it with the appropriate placeholder. For example, replace the name "John Doe"
# with "[NAME]".
#
# Hi John,
#
# I'm writing to you because I noticed you recently purchased a new car. I'm a salesperson
# at a local dealership (Cheap Dealz), and I wanted to let you know that we have a great deal on a new
# car. If you're interested, please let me know.
#
# Thanks,
#
# Jimmy Smith
#
# Phone: 410-805-2345
# Email: jimmysmith@cheapdealz.com
# """

# PII 除去プロンプト
instruction_prompt_example2 ="""
以下の営業メールを読んでください。個人識別情報（PII）をすべて除去し、
適切なプレースホルダーに置き換えてください。例えば、「山田太郎」という名前は
「[名前]」に置き換えてください。

田中さん、こんにちは。

最近新しい車を購入されたことに気づき、ご連絡しました。私は地元のディーラー（お得カーズ）の
営業担当で、新車のお得な特別価格をご案内したく存じます。
ご興味がございましたら、ぜひご連絡ください。

よろしくお願いいたします。

鈴木一郎

電話番号: 03-1234-5678
メール: suzuki@otokucars.example.com
"""

call_GPT(instruction_prompt_example2)

## 3、エッセイの評価とフィードバック（Essay Evaluation）

生成 AI は、文法、明瞭さ、一貫性、論証の質、根拠の使用法など、複雑な基準に基づいてエッセイを評価しフィードバックを提供するのに活用できます。

In [ ]:
# --- 元の英語プロンプト（参考用） ---
# instruction_prompt_example3 ="""
# Read the following excerpt from an essay and provide feedback based on the following
# criteria: grammar, clarity, coherence, argument quality, and use of evidence. Provide
# a score from 1-10 for each attribute, along with reasoning for your score.
#
# "Despite the popular belief, there's no solid evidence supporting the idea that video
# games lead to violent behavior. Research on the topic is often contradictory and
# inconclusive. Some studies found a correlation, but correlation don't imply causation.
# So, it's premature to blame video games for violence in society."
# """

# エッセイ評価プロンプト
instruction_prompt_example3 ="""
以下のエッセイの抜粋を読み、次の基準に基づいてフィードバックを提供してください：文法、明瞭さ、一貫性、論証の質、根拠の使用法。各項目について1〜10のスコアと、そのスコアの理由を述べてください。

「一般的な信念に反して、ビデオゲームが暴力的な行動につながるという考えを裏付ける確固たる証拠はない。このテーマに関する研究は矛盾していることが多く、決定的ではない。一部の研究では相関関係が見つかったが、相関関係は因果関係を意味しない。したがって、社会の暴力をビデオゲームのせいにするのは時期尚早である。」
"""

call_GPT(instruction_prompt_example3)

# Role Prompting（ロールプロンプティング）

LLM に役割を割り当てること、つまり**ロールプロンプティング**は、AI が生成するテキストのスタイルを制御するための技法です。数学の問題を解く際の精度向上にも効果があります。

ロールプロンプティングの実装は簡単で、AI に「フードクリティックになりきって」や「探偵として行動して」と指示するだけです。

ロールプロンプティングは広く使われている技法です。

In [ ]:
# --- 元の英語プロンプト（参考用） ---
# role_prompt_example4 ="""
# You are a brilliant mathematician who can solve any problem in the world.
# Attempt to solve the following problem:
# What is (100*100)/(400*56)?
# """

role_prompt_example4 ="""
あなたは世界のどんな問題でも解ける天才数学者です。
以下の問題を解いてください：

(100*100)/(400*56) はいくつですか？
"""

call_GPT(role_prompt_example4)

# Few-Shot Prompting（少数例プロンプティング）

もう一つのプロンプティング技法が **Few-Shot Prompting** です。これは、やりたいことの例（ショット）をいくつかモデルに示す手法です。

この技法のバリエーションとして、**Zero-shot prompting**（例なし）と **One-shot prompting**（例が1つ）があります。

例の構造化の仕方は重要で、一貫性を保つ必要があります。また、例の分布は比較的均等であるべきです。


## 1、出力の構造化（Structured Output）

Few-shot prompting の重要なユースケースの一つは、出力を特定の形式で構造化する必要があるが、その形式をモデルに言葉で説明するのが難しい場合です。具体例を見てみましょう：地元の新聞記事を分析して、近隣の町で知られている市民の名前と職業を集める経済分析を行っているとします。

モデルに各記事を読ませ、`名前 [職業]` の形式で名前と職業のリストを出力させたいとします。これを実現するには、いくつかの例を示します。

In [ ]:
# --- 元の英語プロンプト（参考用） ---
# role_prompt_example5 ="""
# In the bustling town of Emerald Hills, a diverse group of individuals made their mark. Sarah Martinez, a dedicated nurse, ...
# 1. Sarah Martinez [NURSE]
# 2. David Thompson [SOFTWARE ENGINEER]
# ...
# (以下、3 つの町の文章と名前・職業リスト)
# """

# 長文からの情報抽出の例（Few-shot 形式）
role_prompt_example5 ="""
活気あふれるエメラルドヒルズの町では、多様な人々がその足跡を残しました。献身的な看護師のサラ・マルティネスは、地元の病院での思いやりのあるケアで知られていました。革新的なソフトウェアエンジニアのデビッド・トンプソンは、テクノロジー業界に革命をもたらす画期的なプロジェクトに精力的に取り組みました。一方、才能ある画家で壁画家のエミリー・ナカムラは、建物やギャラリーの壁を飾る生き生きとした示唆に富む作品を描きました。最後に、野心的な起業家のマイケル・オコネルは、ユニークでエコフレンドリーなカフェを開き、すぐに町で人気の待ち合わせ場所になりました。これらの人々はそれぞれ、エメラルドヒルズの豊かなコミュニティに貢献しました。
1. サラ・マルティネス [看護師]
2. デビッド・トンプソン [ソフトウェアエンジニア]
3. エミリー・ナカムラ [画家]
4. マイケル・オコネル [起業家]

町の中心部では、シェフのオリバー・ハミルトンが農場直送レストラン「グリーンプレート」で食のシーンを一変させました。オリバーの地元産オーガニック食材へのこだわりは、食評家や地元の人々から絶賛されています。

通りを少し歩くと、リバーサイドグローブ図書館があります。主任司書のエリザベス・チェンは、すべての人にとって居心地の良い包括的な空間を作るために懸命に働いてきました。彼女の蔵書拡充と子供向け読書プログラム設立の取り組みは、町の識字率に大きな影響を与えました。

魅力的な町の広場を散歩すると、壁を飾る美しい壁画に魅了されるでしょう。これらの傑作は著名な画家イザベラ・トレスの作品で、リバーサイドグローブの本質を捉える彼女の才能は町に活気をもたらしました。

リバーサイドグローブの運動面での実績も注目に値します。元オリンピック水泳選手からコーチに転身したマーカス・ジェンキンスのおかげです。マーカスは自身の経験と情熱を活かして町の若者を指導し、リバーサイドグローブ水泳チームを複数の地域大会優勝に導きました。
1. オリバー・ハミルトン [シェフ]
2. エリザベス・チェン [司書]
3. イザベラ・トレス [画家]
4. マーカス・ジェンキンス [コーチ]

オークバレーは、その技術と献身がコミュニティに永続的な影響を残した素晴らしい3人の人物の故郷です。

町の賑やかなファーマーズマーケットでは、美味しくサステナブルに栽培された農産物で知られる情熱的な有機農家、ローラ・シモンズに出会えます。健康的な食事を推進する彼女の献身は、町がよりエコ志向のライフスタイルを受け入れるきっかけとなりました。

オークバレーのコミュニティセンターでは、熟練したダンスインストラクターのケビン・アルバレスが、あらゆる年齢の人々に動く喜びをもたらしました。彼のインクルーシブなダンスクラスは住民の間に一体感と自己表現の場を育み、地元のアート文化を豊かにしました。

最後に、疲れを知らないボランティアのレイチェル・オコナーは、さまざまな慈善活動に時間を捧げています。他者の生活を改善するための彼女の献身は、オークバレーに強いコミュニティ意識を醸成するのに大きく貢献しました。

それぞれのユニークな才能と揺るぎない献身を通じて、ローラ、ケビン、レイチェルはオークバレーの一部となり、活気に満ちた繁栄する小さな町の発展に貢献しました。
"""

call_GPT(role_prompt_example5)

「ショット」は「例」と同義です。Few-shot prompting のほかに、2つの shot prompting のバリエーションがあります。違いは、モデルに示す例の数だけです。

## 2、Zero-shot prompting（ゼロショットプロンプティング）

Zero-shot prompting は最も基本的なプロンプティングの形式です。例を示さずにプロンプトだけをモデルに与え、応答を生成させます。つまり、これまで見てきた指示プロンプトやロールプロンプトはすべて Zero-shot プロンプトです。

In [ ]:
# --- 元の英語プロンプト（参考用） ---
# role_prompt_example6 ="""
# Add 2+2:
# """

# ゼロショット（例なし）の計算
role_prompt_example6 ="""
2+2 を計算してください:
"""

call_GPT(role_prompt_example6)

## 3、One-shot prompting（ワンショットプロンプティング）

One-shot prompting は、モデルに1つだけ例を示す手法です。

In [ ]:
# --- 元の英語プロンプト（参考用） ---
# role_prompt_example7 ="""
# Add 3+3: 6
# Add 2+2:
# """

# ワンショット（1つの例付き）の計算
role_prompt_example7 ="""
3+3 を計算: 6
2+2 を計算:
"""

call_GPT(role_prompt_example7)

# Combining Techniques（技法の組み合わせ）

ここまで複数のプロンプトの種類と、それらを組み合わせる方法を学びました。

In [ ]:
# --- 元の英語プロンプト（参考用） ---
# combined_prompt ="""
# Twitter is a social media platform where users can post short messages called "tweets".
# Tweets can be positive or negative, and we would like to be able to classify tweets as
# positive or negative. Here are some examples of positive and negative tweets. Make sure
# to classify the last tweet correctly.
#
# Q: Tweet: "What a beautiful day!"
# Is this tweet positive or negative?
#
# A: positive
#
# Q: Tweet: "I hate this class"
# Is this tweet positive or negative?
#
# A: negative
#
# Q: Tweet: "I love pockets on jeans"
#
# A:
# """

# コンテキスト、指示、複数の例を含むプロンプト
combined_prompt ="""
Twitter はユーザーが「ツイート」と呼ばれる短いメッセージを投稿できるソーシャルメディアプラットフォームです。
ツイートにはポジティブなものとネガティブなものがあり、ツイートをポジティブかネガティブかに分類したいと考えています。
以下にポジティブなツイートとネガティブなツイートの例を示します。
最後のツイートを正しく分類してください。

Q: ツイート: 「今日はなんて素晴らしい日なんだ！」
このツイートはポジティブですか、ネガティブですか？

A: ポジティブ

Q: ツイート: 「この授業は最悪だ」
このツイートはポジティブですか、ネガティブですか？

A: ネガティブ

Q: ツイート: 「ジーンズのポケットが大好き」

A:
"""

call_GPT(combined_prompt)

# Formalizing Prompting（プロンプティングの定式化）

より複雑なアプリケーション開発のシナリオでは、上記の技法を一定の順序で組み合わせて、タスクの意図に沿って LLM を導くことができます。

おおまかに以下の要素があります:

* ロール（役割）
* 指示 / タスク
* 質問
* コンテキスト（文脈）
* 例（Few-shot）


In [ ]:
# --- 元の英語プロンプト（参考用） ---
# complex_prompt_example1 ="""
# You are a doctor. Read this medical history and predict risks for the patient:
#
# January 1, 2000: Fractured right arm playing basketball. Treated with a cast.
# February 15, 2010: Diagnosed with hypertension. Prescribed lisinopril.
# September 10, 2015: Developed pneumonia. Treated with antibiotics and recovered fully.
# March 1, 2022: Sustained a concussion in a car accident. Admitted to the hospital and monitored for 24 hours.
# """

# 複雑なプロンプトの例 1
# ロール、指示、コンテキストを組み合わせ
complex_prompt_example1 ="""
あなたは医師です。以下の病歴を読み、患者のリスクを予測してください：

2000年1月1日: バスケットボール中に右腕を骨折。ギプスで治療。
2010年2月15日: 高血圧と診断。リシノプリルを処方。
2015年9月10日: 肺炎を発症。抗生物質で治療し、完全に回復。
2022年3月1日: 自動車事故で脳震盪を起こす。入院し、24時間にわたり経過観察。
"""

call_GPT(complex_prompt_example1)